In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


### Funciones auxiliares

Para obtener estación del año como int y zona de la boya 

In [14]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

### Parámetros de los modelos

Cada modelo está con sus hiperparámetros (Nos quedamos con los de una iteración inicial de Optuna en EN, KNN, SVR y XGB, y mantenemos los que pusimos por defecto en CAT, MLP, LBM y RF. Esto lo decidimos después de ver los resultados al entrenar todo con los que nos daba Optuna y comparar resultados modelo a modelo).

In [15]:
model_params ={
    "XGB" : {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 7,
        'min_child_weight': 2,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        #'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
        },

    "LBM" : {
        'learning_rate': 0.04,
        'num_leaves': 20,
        'max_depth': 7,
        'min_child_samples': 4,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'n_estimators': 1000,
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',  
        'verbosity': -1,
        #'early_stopping_rounds': 50
        },

    "MLP": {
        'hidden_layer_sizes': (100,),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.0001,
        'learning_rate': 'constant',
        'learning_rate_init': 0.001,
        'max_iter': 200,
        'shuffle': True,
        'random_state': None,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
        },

    "SVR": {
        'kernel': 'rbf',        
        'C': 9.5,               
        'epsilon': 0.1,           
        'gamma': 'scale',        
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,          
        'verbose': False,
    },

    "KNN": {
        'n_neighbors': 5,
        'weights': 'distance',      
        'algorithm': 'auto',      
        'leaf_size': 25,
        'p': 2,                    
        'metric': 'minkowski',
        'n_jobs': -1             
    },

    "RF": {
        'n_estimators': 100,         
        'criterion': 'squared_error',
        'max_depth': 10,         
        'min_samples_split': 2,
        'min_samples_leaf': 2,    
        'bootstrap': True,
        'random_state': 42,
        'verbose': 0
    },

    "CAT": {
        'iterations': 1000,
        'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'allow_writing_files': False,
        'early_stopping_rounds': 50,
        'verbose': False
    },

    "ELN": {
        'alpha': 0.2,             
        'l1_ratio': 0.5,           
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }

}


models = {
    "XGB": XGBRegressor(**model_params['XGB']),
    "LBM": LGBMRegressor(**model_params['LBM']),
    "MLP": MLPRegressor(**model_params['MLP']),
    "SVR": SVR(**model_params['SVR']),
    "KNN": KNeighborsRegressor(**model_params['KNN']),
    "LR": LinearRegression(),
    "RF": RandomForestRegressor(**model_params['RF']),
    "CAT": CatBoostRegressor(**model_params["CAT"]),
    "ELN":  ElasticNet(**model_params["ELN"])
}

### Función para hacer validación cruzada

- Se define el número de folds, 5.
- Para cada uno de los datasets (conjunto de datos según la profundidad de la Chl que estimamos, tipo de procesado y ventana de agregación):
    - Split de train y test en 75/25. Estratificado para tener el mismo número de muestras con Chl alta (>5).
    - Separamos X e y. Aseguramos que ni el target ni el indicador de Chl alta estén en X para evitar data leakage.
    - Definimos los dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble. También otro dict para test.
    - Definimos el dict para guardar los resultados de las métricas (R2 y RMSE).
    - Definimos el StratifiedKFold con los 5 folds, utilizando la clase de clorofila alta.
    - Hacemos el loop sobre cada uno de los modelos.
        - Separamos en conjunto de entrenamiento y validación y:
            - si el modelo está basado en distancias escalamos los datos, entrenamos el modelo y hacemos la transformada inversa.
            - si el modelo está basado en decisiones, entrenamos directamente y hacemos las predicciones.
        - Guardamos las predicciones sobre validación, las y's correspondientes y los índices de validación para poder usarlos en el ensemble.
        - Guardamos la predicción de test con 1/5 del peso en cada fold. Así al final tendremos la media de los 5 folds.
        - Calculamos y guardamos métricas sobre validación.
    - Extendemos el dict de resultados para guardar las métricas del ensemble
    - Hacemos otro loop de 5 folds:
        - Al llamar a cada fold, usamos los otros 4 para entrenar el meta-modelo y el fold actual para la evaluación. Así tenemos, igual que con el resto de modelos, 5 medidas sobre validación sobre las que hacer la media y desviación estándar.
    - Hacemos otro loop sobre cada modelo para calcular las métricas sobre el conjunto de test
- Guardamos el dict de resultados con pickle.

In [47]:
def cross_validation_training(dfs, depth):

    FOLDS = 5
    # Para hacer clip a las predicciones y forzar > 0
    correct = True
    results = {}

    for nombre_df, df in list(dfs.items()):
    #for nombre_df, df in islice(dfs.items(), 3):
        print(f"\n=== Procesando {nombre_df} ===")
        # Ignoramos las columnas de Date, Lat, Lon y Buoy
        df = df.iloc[:, 4:]

        # Separamos el conjunto de datos en train y test: Train 75% Test 25%
        train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])
        # Seleccionamos la columna que queremos predecir
        target = "Chl"

        # Quitamos esa columna y el indicador de clorofila alta
        X = train.drop(columns=[target, "High_Chl", "Turbidez"])
        # Para y cogemos solamente Chl
        y = train[target]
        # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
        y_class = train["High_Chl"]

        # Definimos X e y para test
        X_test = test.drop(columns=[target, "High_Chl", "Turbidez"])
        y_test = test[target]

        # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
        val_preds = {name: np.zeros(len(train)) for name in models}
        y_vals = defaultdict(list)
        val_indices = {}
        # Dict para guardar las predicciones sobre test
        test_preds = {name: np.zeros(len(test)) for name in models}
        
        # Dict para guardar resultados
        results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}
        # Stratified KFold de 5 folds
        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

        # Loop para entrenar cada uno de los modelos
        for name, model in models.items():
            print(f"\n=== Training {name} ===")
            # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
                print(f"Fold {fold+1}")
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                # Para modelos basados en distancias escalamos los datos
                if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                    # Escalado dentro del loop de folds para evitar data leakage entre folds
                    scaler_X = RobustScaler()
                    scaler_y = RobustScaler()
                    X_train_scaled = scaler_X.fit_transform(X_train)
                    X_val_scaled = scaler_X.transform(X_val)
                    X_test_scaled = scaler_X.transform(X_test)
                    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                    # Entrenamos modelo con datos escalados
                    model.fit(X_train_scaled, y_train_scaled)
                    # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                    val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                    test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
                # Para modelos basados en árboles no es necesario escalar
                else:
                    # Entrenamos el modelo
                    model.fit(X_train, y_train)
                    # Predicción sobre val y test
                    val_pred = model.predict(X_val)
                    test_pred = model.predict(X_test)

                if correct:
                    val_pred = np.clip(val_pred, 0.3, None)
                    test_pred = np.clip(test_pred, 0.3, None)

                # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
                val_preds[name][val_idx] = val_pred
                if name == list(models.keys())[0]:
                    # Solo lo guardamos una vez
                    y_vals[fold] = y_val
                    val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

                # Guardamos la predicción de test, haciendo la media entre los folds
                test_preds[name] += test_pred / FOLDS

                # Calculamos y guardamos métricas
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                r2 = r2_score(y_val, val_pred)
                results[nombre_df][name]['RMSE'].append(rmse)
                results[nombre_df][name]['R2'].append(r2)

        # Extendemos el dict de resultados con el ensemble
        results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

        for fold in range(FOLDS):
            # Índices y valores del fold actual
            fold_val_idx = val_indices[fold]
            meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
            meta_y_val = y_vals[fold]

            # Índices de entrenamiento: todos menos el fold actual
            train_folds = [i for i in range(FOLDS) if i != fold]
            train_idx = np.concatenate([val_indices[i] for i in train_folds])
            meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
            meta_y_train = y.iloc[train_idx]

            # Entrenamos el meta-modelo solo con los otros 4 folds
            meta_model = Ridge().fit(meta_X_train, meta_y_train)

            # Predicción en el fold actual (no visto)
            ensemble_pred = meta_model.predict(meta_X_val)
            if correct:
                ensemble_pred = np.clip(ensemble_pred, 0.3, None)

            rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
            r2 = r2_score(meta_y_val, ensemble_pred)
            results[nombre_df]["ENS"]['RMSE'].append(rmse)
            results[nombre_df]["ENS"]['R2'].append(r2)



    # === Evaluación final sobre test ===
        for name in models:
            rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
            r2_test = r2_score(y_test, test_preds[name])
            results[nombre_df][name]["RMSE test"] = rmse_test
            results[nombre_df][name]["R2 test"] = r2_test

        # Construcción del meta-modelo sobre todo el conjunto de validación
        final_meta_X = np.vstack([val_preds[model] for model in models]).T
        final_meta_y = y.values
        ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

        # Predicción sobre test del ensemble
        meta_X_test = np.vstack([test_preds[model] for model in models]).T
        ensemble_test_pred = ensemble_model.predict(meta_X_test)
        if correct:
            ensemble_test_pred = np.clip(ensemble_test_pred, 0.3, None)
        # Evaluación del ensemble sobre test
        rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
        r2_ens_test = r2_score(y_test, ensemble_test_pred)
        results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
        results[nombre_df]["ENS"]["R2 test"] = r2_ens_test

    with open(f"training_results/results_entrenamiento_CV_corrected_{depth}.pkl", "wb") as f:
        pickle.dump(results, f)

    return results

### Experimentos por profundidad

Para cada una de las profundidades de interés, hacemos todo el proceso que incluye las funciones anteriores y el entrenamiendo con validación cruzada + evaluación en test.

In [48]:
depths = ["eq_0", "eq_1", "in_0_1", "in_1_2", "in_2_3", "in_3_4", "gt_4"]
for depth in depths: 
    # Cargamos los csv de los tifs
    path = "saved_files/dataset"
    dfs = {}
    all_datasets = False
    for archivo in os.listdir(path):
        if all_datasets:
            if f"{depth}" in archivo:
                nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
                ruta_completa = os.path.join(path, archivo)
                dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)
        else:
            if archivo.endswith(f"{depth}_features.csv"):
                nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
                ruta_completa = os.path.join(path, archivo)
                dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown","rtoa"]:
            dfs[nombre_df] = df.dropna()

    
    for nombre_df, df in dfs.items():
        # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
        df["High_Chl"] = df["Chl"]>5
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        df['Season'] = df['Date'].dt.month.apply(get_season)
        
        # Etiquetamos la zona de la observación (comentado porque para aplicar el modelo habría que segmentar todo el Mar Menor - se puede hacer por px)
        # df['Zone'] = df['Buoy'].apply(get_zone)
        # Ponemos las columnas como categóricas, para Season y Zone
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].astype('category')

        df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df

    results = cross_validation_training(dfs, depth)


=== Procesando C2X-Complex_rhow_9x9_depth_eq_0 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 3
Fold 4


...

Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_9x9_depth_gt_4 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X_rhown_5x5_depth_gt_4 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5



/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2RCC_rhown_9x9_depth_gt_4 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X_rhow_5x5_depth_gt_4 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
F

In [52]:
depth = "in_2_3"

with open(f"training_results/results_entrenamiento_CV_corrected_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [53]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [12]:
df_results["R2"]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_15x15_depth_in_0_1,0.69 ± 0.17,0.46 ± 0.07,0.69 ± 0.12,0.71 ± 0.10,0.54 ± 0.31,0.25 ± 0.81,0.67 ± 0.12,0.60 ± 0.23,0.68 ± 0.11,0.62 ± 0.24
C2RCC_rhow_1x1_depth_in_0_1,0.76 ± 0.08,0.45 ± 0.07,0.76 ± 0.11,0.75 ± 0.10,0.73 ± 0.06,0.51 ± 0.14,0.68 ± 0.09,0.66 ± 0.11,0.68 ± 0.07,0.76 ± 0.05
C2RCC_rhow_3x3_depth_in_0_1,0.73 ± 0.03,0.41 ± 0.06,0.73 ± 0.06,0.75 ± 0.06,0.71 ± 0.08,0.36 ± 0.35,0.54 ± 0.14,0.65 ± 0.06,0.70 ± 0.05,0.74 ± 0.03
C2RCC_rhow_5x5_depth_in_0_1,0.74 ± 0.08,0.49 ± 0.05,0.65 ± 0.12,0.74 ± 0.08,0.70 ± 0.09,0.56 ± 0.16,0.56 ± 0.20,0.67 ± 0.10,0.73 ± 0.04,0.71 ± 0.07
C2RCC_rhow_9x9_depth_in_0_1,0.76 ± 0.09,0.47 ± 0.06,0.71 ± 0.11,0.76 ± 0.10,0.63 ± 0.11,0.55 ± 0.10,0.52 ± 0.22,0.68 ± 0.08,0.72 ± 0.06,0.72 ± 0.06
C2RCC_rhown_15x15_depth_in_0_1,0.71 ± 0.15,0.46 ± 0.06,0.65 ± 0.18,0.68 ± 0.13,0.57 ± 0.29,-0.02 ± 1.18,0.57 ± 0.21,0.60 ± 0.22,0.67 ± 0.11,0.63 ± 0.19
C2RCC_rhown_1x1_depth_in_0_1,0.76 ± 0.06,0.43 ± 0.07,0.74 ± 0.09,0.74 ± 0.05,0.69 ± 0.10,0.53 ± 0.20,0.51 ± 0.24,0.70 ± 0.09,0.65 ± 0.07,0.78 ± 0.06
C2RCC_rhown_3x3_depth_in_0_1,0.76 ± 0.03,0.41 ± 0.06,0.72 ± 0.04,0.74 ± 0.08,0.73 ± 0.07,0.36 ± 0.19,0.59 ± 0.15,0.68 ± 0.07,0.68 ± 0.05,0.78 ± 0.03
C2RCC_rhown_5x5_depth_in_0_1,0.76 ± 0.08,0.48 ± 0.05,0.67 ± 0.09,0.73 ± 0.10,0.75 ± 0.09,0.54 ± 0.21,0.71 ± 0.02,0.69 ± 0.05,0.72 ± 0.04,0.74 ± 0.08
C2RCC_rhown_9x9_depth_in_0_1,0.76 ± 0.09,0.47 ± 0.05,0.67 ± 0.13,0.75 ± 0.11,0.67 ± 0.08,0.50 ± 0.25,0.51 ± 0.16,0.69 ± 0.07,0.71 ± 0.05,0.69 ± 0.07


In [13]:
df_results["R2 test"]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_15x15_depth_in_0_1,0.75,0.51,0.80,0.84,0.72,0.61,0.74,0.75,0.69,0.77
C2RCC_rhow_1x1_depth_in_0_1,0.74,0.53,0.74,0.80,0.66,0.38,0.73,0.69,0.62,0.78
C2RCC_rhow_3x3_depth_in_0_1,0.81,0.55,0.78,0.81,0.72,0.58,0.69,0.77,0.66,0.82
C2RCC_rhow_5x5_depth_in_0_1,0.81,0.56,0.82,0.85,0.79,0.43,0.68,0.76,0.70,0.83
C2RCC_rhow_9x9_depth_in_0_1,0.81,0.53,0.82,0.81,0.80,0.65,0.67,0.77,0.71,0.83
C2RCC_rhown_15x15_depth_in_0_1,0.76,0.51,0.79,0.83,0.71,0.61,0.67,0.73,0.68,0.76
C2RCC_rhown_1x1_depth_in_0_1,0.75,0.52,0.82,0.78,0.64,0.27,0.61,0.68,0.62,0.76
C2RCC_rhown_3x3_depth_in_0_1,0.82,0.55,0.79,0.80,0.75,0.54,0.71,0.75,0.65,0.81
C2RCC_rhown_5x5_depth_in_0_1,0.83,0.56,0.84,0.84,0.80,0.50,0.77,0.79,0.70,0.83
C2RCC_rhown_9x9_depth_in_0_1,0.81,0.53,0.79,0.81,0.80,0.68,0.59,0.77,0.72,0.82


In [54]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")

In [50]:
df_sorted.iloc[:10].index

Index(['TOA_9x9_depth_in_3_4', 'TOA_3x3_depth_in_3_4', 'TOA_5x5_depth_in_3_4',
       'C2X-Complex_rhow_5x5_depth_in_3_4', 'TOA_1x1_depth_in_3_4',
       'TOA_15x15_depth_in_3_4', 'C2X-Complex_rhown_5x5_depth_in_3_4',
       'C2X-Complex_rhow_9x9_depth_in_3_4',
       'C2X-Complex_rhow_15x15_depth_in_3_4',
       'C2X-Complex_rhown_9x9_depth_in_3_4'],
      dtype='object')

In [51]:
## 3-4
df_sorted.iloc[:10]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_9x9_depth_in_3_4,0.58,0.35,0.57,0.62,0.43,0.38,0.70,0.61,0.47,0.55
TOA_3x3_depth_in_3_4,0.58,0.34,0.49,0.62,0.62,0.40,0.67,0.66,0.46,0.55
TOA_5x5_depth_in_3_4,0.57,0.35,0.52,0.61,0.65,0.45,0.59,0.63,0.42,0.55
C2X-Complex_rhow_5x5_depth_in_3_4,0.60,0.50,0.55,0.55,0.62,0.42,0.63,0.65,0.53,0.63
TOA_1x1_depth_in_3_4,0.54,0.33,0.47,0.60,0.39,0.18,0.64,0.55,0.43,0.48
TOA_15x15_depth_in_3_4,0.64,0.36,0.52,0.62,0.53,0.38,0.62,0.62,0.47,0.58
C2X-Complex_rhown_5x5_depth_in_3_4,0.58,0.51,0.58,0.55,0.60,0.35,0.63,0.64,0.49,0.62
C2X-Complex_rhow_9x9_depth_in_3_4,0.58,0.48,0.57,0.51,0.60,0.47,0.64,0.62,0.59,0.60
C2X-Complex_rhow_15x15_depth_in_3_4,0.50,0.48,0.44,0.42,0.46,0.41,0.62,0.55,0.53,0.50
C2X-Complex_rhown_9x9_depth_in_3_4,0.58,0.49,0.56,0.54,0.52,0.38,0.61,0.60,0.57,0.59


In [55]:
## 2-3
df_sorted.iloc[:10]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_2_3,0.73,0.19,0.81,0.78,0.65,0.21,0.62,0.64,0.52,0.64
TOA_9x9_depth_in_2_3,0.70,0.18,0.77,0.77,0.69,0.27,0.61,0.65,0.52,0.63
TOA_5x5_depth_in_2_3,0.76,0.18,0.75,0.75,0.75,0.36,0.63,0.68,0.50,0.74
C2X-Complex_rhow_5x5_depth_in_2_3,0.71,0.21,0.73,0.58,0.60,0.24,0.44,0.59,0.74,0.66
C2RCC_rhown_5x5_depth_in_2_3,0.72,0.37,0.72,0.74,0.67,0.61,0.64,0.64,0.64,0.68
TOA_3x3_depth_in_2_3,0.73,0.17,0.72,0.73,0.73,0.35,0.62,0.65,0.48,0.70
C2X-Complex_rhown_5x5_depth_in_2_3,0.66,0.25,0.68,0.54,0.54,-0.48,0.35,0.55,0.73,0.60
C2RCC_rhow_3x3_depth_in_2_3,0.68,0.31,0.70,0.72,0.64,0.55,0.66,0.61,0.65,0.61
C2X-Complex_rhown_9x9_depth_in_2_3,0.72,0.43,0.64,0.61,0.65,0.41,0.58,0.65,0.70,0.65
C2X_rhow_9x9_depth_in_2_3,0.69,0.49,0.67,0.59,0.61,0.64,0.71,0.62,0.67,0.66


In [36]:
## 1-2
df_sorted.iloc[:10]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_5x5_depth_in_1_2,0.86,0.59,0.86,0.79,0.84,0.71,0.69,0.75,0.87,0.84
C2X_rhow_3x3_depth_in_1_2,0.83,0.67,0.72,0.80,0.86,0.68,0.72,0.77,0.67,0.82
C2X-Complex_rhown_5x5_depth_in_1_2,0.86,0.62,0.85,0.78,0.80,0.71,0.68,0.72,0.86,0.78
C2X-Complex_rhow_9x9_depth_in_1_2,0.86,0.54,0.85,0.76,0.82,0.53,0.68,0.79,0.83,0.82
C2X-Complex_rhow_3x3_depth_in_1_2,0.80,0.56,0.77,0.77,0.77,0.54,0.70,0.69,0.84,0.77
C2X-Complex_rhown_3x3_depth_in_1_2,0.81,0.58,0.84,0.75,0.74,0.67,0.70,0.71,0.83,0.78
C2RCC_rhown_3x3_depth_in_1_2,0.81,0.49,0.83,0.82,0.76,0.62,0.70,0.72,0.79,0.77
C2X-Complex_rhown_9x9_depth_in_1_2,0.81,0.58,0.78,0.76,0.70,0.48,0.60,0.70,0.83,0.73
C2X-Complex_rhow_15x15_depth_in_1_2,0.82,0.53,0.79,0.74,0.83,-0.44,0.68,0.80,0.80,0.77
C2X_rhow_5x5_depth_in_1_2,0.82,0.69,0.74,0.75,0.78,0.72,0.73,0.78,0.68,0.81


In [42]:
## 0-1
df_sorted.iloc[:10]

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_9x9_depth_in_0_1,0.89,0.61,0.88,0.83,0.88,0.65,0.69,0.83,0.74,0.89
TOA_15x15_depth_in_0_1,0.67,0.36,0.86,0.80,0.60,0.51,0.65,0.57,0.32,0.62
C2X-Complex_rhown_9x9_depth_in_0_1,0.86,0.61,0.83,0.83,0.84,0.65,0.70,0.80,0.74,0.84
C2X-Complex_rhow_5x5_depth_in_0_1,0.85,0.60,0.83,0.82,0.84,-0.11,0.73,0.80,0.72,0.84
C2RCC_rhow_5x5_depth_in_0_1,0.81,0.56,0.82,0.85,0.79,0.43,0.68,0.76,0.70,0.83
C2RCC_rhow_15x15_depth_in_0_1,0.75,0.51,0.80,0.84,0.72,0.61,0.74,0.75,0.69,0.77
C2X-Complex_rhown_15x15_depth_in_0_1,0.76,0.58,0.83,0.84,0.71,0.64,0.71,0.74,0.72,0.76
C2RCC_rhown_5x5_depth_in_0_1,0.83,0.56,0.84,0.84,0.80,0.50,0.77,0.79,0.70,0.83
C2X-Complex_rhow_15x15_depth_in_0_1,0.83,0.58,0.82,0.84,0.84,0.68,0.56,0.79,0.72,0.82
C2RCC_rhow_9x9_depth_in_0_1,0.81,0.53,0.82,0.81,0.80,0.65,0.67,0.77,0.71,0.83


In [24]:
df_sorted = df_results["RMSE test"].copy()
df_sorted["min_RMSE"] = df_sorted.min(axis=1)
df_sorted = df_sorted.sort_values("min_RMSE", ascending=True)
df_sorted = df_sorted.drop(columns="min_RMSE")